In [ ]:
import torch
from transformers import PatchTSTConfig, PatchTSTForRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Embedding, Dense, concatenate, RepeatVector, Input, Bidirectional, RepeatVector, TimeDistributed, GRU

from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense


import numpy as np
import pandas as pd

from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
merged_df = pd.read_csv("/content/drive/MyDrive/")

In [ ]:
merged_df.head(3)

In [ ]:
dates = pd.to_datetime(merged_df['Date'])
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

In [ ]:
merged_df = merged_df.drop(columns=['Fluctuations'])
merged_df

In [ ]:
target = merged_df['Close'].values

In [ ]:
cols = list(merged_df)[4:]
data = merged_df[cols].astype(float)

In [ ]:
def minmax_scaler(data):
    min_val = data.min(axis=0)
    max_val = data.max(axis=0)
    scaled_data = (data - min_val) / (max_val - min_val)
    return np.array(scaled_data), min_val, max_val

def minmax_inverse_transform(scaled_data, min_val, max_val):
    return scaled_data * (max_val - min_val) + min_val

In [ ]:
data_scaled, min_val, max_val = minmax_scaler(data)

In [ ]:
n_train = int(0.8*data_scaled.shape[0])
train_data_scaled = data_scaled[0: n_train]
train_dates = dates[0: n_train]

test_data_scaled = data_scaled[n_train:]
test_dates = dates[n_train:]

In [ ]:
n_train

In [ ]:
test_dates.shape

In [ ]:
train_data_scaled.shape, test_data_scaled.shape

In [ ]:
def reformat_data_for_LSTM(train_data_scaled, test_data_scaled, seq_len, pred_days):
    trainX = []
    trainY = []
    testX = []
    testY = []
    n_train = len(train_data_scaled)

    for i in range(seq_len, n_train - pred_days + 1):
        trainX.append(train_data_scaled[i - seq_len:i, 0:train_data_scaled.shape[1]])
        trainY.append(train_data_scaled[i + pred_days - 1:i + pred_days, 0])

    for i in range(seq_len, len(test_data_scaled) - pred_days + 1):
        testX.append(test_data_scaled[i - seq_len:i, 0:test_data_scaled.shape[1]])
        testY.append(test_data_scaled[i + pred_days - 1:i + pred_days, 0])

    trainX, trainY = np.array(trainX), np.array(trainY)
    testX, testY = np.array(testX), np.array(testY)

    return trainX, trainY, testX, testY

trainX_3, trainY_3, testX_3, testY_3 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 3, 1)
trainX_5, trainY_5, testX_5, testY_5 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 5, 1)
trainX_4, trainY_4, testX_4, testY_4 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 4, 1)
# trainX_30, trainY_30, testX_30, testY_30 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 30, 1)
# trainX_60, trainY_60, testX_60, testY_60 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 60, 1)
# trainX_120, trainY_120, testX_120, testY_120 = reformat_data_for_LSTM(train_data_scaled, test_data_scaled, 120, 1)

In [ ]:
input_shape = (trainX_3.shape[1], trainX_3.shape[2])
output_shape = trainY_3.shape[1]

In [ ]:
learning_rate=0.005
BATCH_SIZE = 64

def create_Bi_LSTM_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Bidirectional(LSTM(64, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(LSTM(32, return_sequences=False)))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_Bi_RNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Bidirectional(SimpleRNN(64, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(32, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(16, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(8, return_sequences=False)))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def create_LSTM_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(32, return_sequences=False))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_CNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Conv1D(64, kernel_size=1, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Conv1D(32, kernel_size=1, activation='relu'))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Flatten())
    model.add(Dense(32, activation='relu'))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def create_RNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(SimpleRNN(64, return_sequences=True, input_shape=input_shape))
    model.add(SimpleRNN(32, return_sequences=False))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def MAPE(y_test, y_pred):
	return np.mean(np.abs((y_test - y_pred) / y_test)) * 100

def evaluate_model(y_test, y_pred):
    mape = MAPE(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return {
        'MAPE': mape,
        'RMSE': rmse,
        'MAE': mae,
        'R^2': r2
    }

In [ ]:
from transformers import PatchTSTConfig, PatchTSTModel
from transformers import AutoConfig

configuration = PatchTSTConfig()
model = PatchTSTModel(configuration)
configuration = model.config

In [ ]:
from transformers import PatchTSTForRegression, PatchTSTConfig
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

config = PatchTSTConfig(
    context_length=3,
    prediction_length=1,
    num_input_channels=19,
    ff_dropout=0.2,
)

model = PatchTSTForRegression(config)

print("Train data shape:", trainX_3.shape)
print("Test data shape:", testX_3.shape)

trainX_3_tensor = torch.tensor(trainX_3, dtype=torch.float32).permute(0, 2, 1)
trainY_3_tensor = torch.tensor(trainY_3, dtype=torch.float32)

train_dataset = TensorDataset(trainX_3_tensor, trainY_3_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = torch.nn.MSELoss()

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_inputs, batch_targets in train_loader:
        batch_inputs = batch_inputs.permute(0, 2, 1)

        outputs = model(past_values=batch_inputs)
        regression_outputs = outputs.regression_outputs

        loss = criterion(regression_outputs.squeeze(-1), batch_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")

    scheduler.step(avg_loss)


In [ ]:
import torch
import torch.nn.functional as F
testX_3_tensor = torch.tensor(testX_3, dtype=torch.float32).permute(0, 2, 1)
testY_3_tensor = torch.tensor(testY_3, dtype=torch.float32)

model.eval()
with torch.no_grad():
    predictions = []
    for i in range(0, len(testX_3_tensor), 64):
        batch_inputs = testX_3_tensor[i:i+64]
        batch_targets = testY_3_tensor[i:i+64]
        print(batch_inputs.shape)
        print(batch_targets.shape)

        if batch_inputs.size(0) < 64:
            padding_size = 64 - batch_inputs.size(0)
            batch_inputs = torch.cat([batch_inputs, batch_inputs[:padding_size]], dim=0)
            batch_targets = torch.cat([batch_targets, batch_targets[:padding_size]], dim=0)

        batch_inputs_padded = batch_inputs.permute(0, 2, 1)
        print(batch_inputs_padded.shape)

        outputs = model(past_values=batch_inputs_padded)
        regression_outputs = outputs.regression_outputs

        predictions.append(regression_outputs)

    predictions = torch.cat(predictions, dim=0).numpy()


In [ ]:
def adjusted_MAPE(y_true, y_pred):
    if y_true.shape != y_pred.shape:
        y_pred = y_pred[:y_true.shape[0]]

    epsilon = 1e-10
    return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

def adjust_shape(y_true, y_pred):
    if y_true.shape != y_pred.shape:
        y_pred = y_pred[:y_true.shape[0]]
    return y_true, y_pred

def adjusted_rmse(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return mean_squared_error(y_true, y_pred)

def adjusted_mae(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return mean_absolute_error(y_true, y_pred)

def adjusted_r2(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return r2_score(y_true, y_pred)

mape_patchtst = adjusted_MAPE(testY_3, predictions)
rmse_patchtst = adjusted_rmse(testY_3, predictions)
mae_patchtst = adjusted_mae(testY_3, predictions)
r2_patchtst = adjusted_r2(testY_3, predictions)

print("Evaluation Metrics for PatchTST:")
print(f"MAPE: {mape_patchtst:.2f}%")
print(f"MSE: {rmse_patchtst:.4f}")
print(f"MAE: {mae_patchtst:.4f}")
print(f"R^2: {r2_patchtst:.4f}")

In [ ]:
rnn_model_3 = create_RNN_model(input_shape, output_shape)
history_rnn = rnn_model_3.fit(trainX_3, trainY_3, epochs=10, batch_size=BATCH_SIZE, verbose=0)
rnn_li = []

prediction_3_rnn = rnn_model_3.predict(testX_3)
rnn_li.append(prediction_3_rnn)

mape_rnn = MAPE(testY_3, prediction_3_rnn)
results_rnn = evaluate_model(testY_3, prediction_3_rnn)


print("Evaluation Metrics for RNN:")
print(f"MAPE: {results_rnn['MAPE']:.2f}%")
print(f"RMSE: {results_rnn['RMSE']:.4f}")
print(f"MAE: {results_rnn['MAE']:.4f}")
print(f"R^2: {results_rnn['R^2']:.4f}")

In [ ]:
lstm_model_3 = create_LSTM_model(input_shape, output_shape)
history_lstm = lstm_model_3.fit(trainX_3, trainY_3, epochs=10, batch_size=BATCH_SIZE, verbose=0)
lstm_li = []

prediction_3_lstm = lstm_model_3.predict(testX_3)
lstm_li.append(prediction_3_lstm)

mape_bi_lstm = MAPE(testY_3, prediction_3_lstm)
results_lstm = evaluate_model(testY_3, prediction_3_lstm)
mape_lstm = MAPE(testY_3, prediction_3_lstm)

print("Evaluation Metrics for LSTM:")
print(f"MAPE: {results_lstm['MAPE']:.2f}%")
print(f"RMSE: {results_lstm['RMSE']:.4f}")
print(f"MAE: {results_lstm['MAE']:.4f}")
print(f"R^2: {results_lstm['R^2']:.4f}")


In [ ]:
cnn_model_3 = create_CNN_model(input_shape, output_shape)
history_cnn = cnn_model_3.fit(trainX_3, trainY_3, epochs=10, batch_size=BATCH_SIZE, verbose=0)
cnn_li = []

prediction_3_cnn = cnn_model_3.predict(testX_3)
cnn_li.append(prediction_3_cnn)

mape_rnn = MAPE(testY_3, prediction_3_rnn)
results_cnn = evaluate_model(testY_3, prediction_3_cnn)
mape_cnn = MAPE(testY_3, prediction_3_cnn)

print("Evaluation Metrics for CNN:")
print(f"MAPE: {results_cnn['MAPE']:.2f}%")
print(f"RMSE: {results_cnn['RMSE']:.4f}")
print(f"MAE: {results_cnn['MAE']:.4f}")
print(f"R^2: {results_cnn['R^2']:.4f}")

In [ ]:
bi_rnn_model_3 = create_Bi_RNN_model(input_shape, output_shape)
history = bi_rnn_model_3.fit(trainX_3, trainY_3, epochs=10, batch_size=BATCH_SIZE,
                verbose=0)
birnn_li = []

prediction_3_bi_rnn = bi_rnn_model_3.predict(testX_3)
birnn_li.append(prediction_3_bi_rnn)

prediction_3_bi_rnn = bi_rnn_model_3.predict(testX_3)
results = evaluate_model(testY_3, prediction_3_bi_rnn)
mape_bi_rnn = MAPE(testY_3, prediction_3_bi_rnn)

print("Evaluation Metrics for Bi-RNN:")
print(f"MAPE: {results['MAPE']:.2f}%")
print(f"RMSE: {results['RMSE']:.4f}")
print(f"MAE: {results['MAE']:.4f}")
print(f"R^2: {results['R^2']:.4f}")

In [ ]:
bi_lstm_model_3 = create_Bi_LSTM_model(input_shape, output_shape)
history = bi_lstm_model_3.fit(trainX_3, trainY_3, epochs=10, batch_size=BATCH_SIZE,
                verbose=0)
bilstm_li = []

prediction_3_bi_lstm = bi_lstm_model_3.predict(testX_3)
bilstm_li.append(prediction_3_bi_lstm)

prediction_3_bi_lstm = bi_lstm_model_3.predict(testX_3)
results = evaluate_model(testY_3, prediction_3_bi_lstm)
mape_bi_lstm = MAPE(testY_3, prediction_3_bi_lstm)

print("Evaluation Metrics for Bi-LSTM:")
print(f"MAPE: {results['MAPE']:.2f}%")
print(f"RMSE: {results['RMSE']:.4f}")
print(f"MAE: {results['MAE']:.4f}")
print(f"R^2: {results['R^2']:.4f}")

In [ ]:
from sklearn.svm import SVR

X_flattened = trainX_3.reshape(trainX_3.shape[0], -1)

model = SVR(kernel='rbf', C=100, gamma=0.03, epsilon=0.1)

model.fit(X_flattened, trainY_3)

svr_li = []
prediction_3_SVR = model.predict(testX_3.reshape(testX_3.shape[0],-1))
svr_li.append(prediction_3_SVR)

results = evaluate_model(testY_3, prediction_3_SVR)
mape_SVR = MAPE(testY_3, prediction_3_SVR)

print("Evaluation Metrics for SVR:")
print(f"MAPE: {results['MAPE']:.2f}%")
print(f"RMSE: {results['RMSE']:.4f}")
print(f"MAE: {results['MAE']:.4f}")
print(f"R^2: {results['R^2']:.4f}")

In [ ]:
import xgboost as xgb
X_flattened = trainX_3.reshape(trainX_3.shape[0], -1)

xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.08, gamma=0, subsample=0.75, colsample_bytree=1, max_depth=7)

xgb_model.fit(X_flattened, trainY_3)

xgb_li = []
prediction_3_XGBOOST = xgb_model.predict(testX_3.reshape(testX_3.shape[0],-1))
xgb_li.append(prediction_3_XGBOOST)
mape_XGBOOST = MAPE(testY_3, prediction_3_XGBOOST)
results = evaluate_model(testY_3, prediction_3_XGBOOST)

print("Evaluation Metrics for XGBoost:")
print(f"MAPE: {results['MAPE']:.2f}%")
print(f"RMSE: {results['RMSE']:.4f}")
print(f"MAE: {results['MAE']:.4f}")
print(f"R^2: {results['R^2']:.4f}")

In [ ]:
import lightgbm as lgb

X_flattened = trainX_3.reshape(trainX_3.shape[0], -1)

lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.08, subsample=0.75, colsample_bytree=1, max_depth=7)

lgb_model.fit(X_flattened, trainY_3)

lgb_li = []
prediction_3_LGBM = lgb_model.predict(testX_3.reshape(testX_3.shape[0], -1))
lgb_li.append(prediction_3_LGBM)

results = evaluate_model(testY_3, prediction_3_LGBM)
mape_LGBM = MAPE(testY_3, prediction_3_LGBM)

print("Evaluation Metrics for LightGBM:")
print(f"MAPE: {results['MAPE']:.2f}%")
print(f"RMSE: {results['RMSE']:.4f}")
print(f"MAE: {results['MAE']:.4f}")
print(f"R^2: {results['R^2']:.4f}")


In [ ]:
prediction_3_SVR_rev = minmax_inverse_transform(prediction_3_SVR, min_val[0], max_val[0])
prediction_3_bi_lstm_rev = minmax_inverse_transform(prediction_3_bi_lstm, min_val[0], max_val[0])
prediction_3_bi_rnn_rev = minmax_inverse_transform(prediction_3_bi_rnn, min_val[0], max_val[0])
prediction_3_XGBOOST_rev = minmax_inverse_transform(prediction_3_XGBOOST, min_val[0], max_val[0])
prediction_3_rnn_rev = minmax_inverse_transform(prediction_3_rnn, min_val[0], max_val[0])
prediction_3_lstm_rev = minmax_inverse_transform(prediction_3_lstm, min_val[0], max_val[0])
prediction_3_cnn_rev = minmax_inverse_transform(prediction_3_cnn, min_val[0], max_val[0])
prediction_3_lgbm_rev = minmax_inverse_transform(prediction_3_LGBM, min_val[0], max_val[0])

plt.figure(figsize=(20, 10))
plt.xlabel('Date', fontsize=14)
plt.ylabel('S&P500', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.plot(test_dates[5:], target[-367:], color='black', linestyle='--', label='Actual Close Price')
plt.plot(test_dates[5:], prediction_3_SVR_rev[-367:], color='red', label='SVR')
plt.plot(test_dates[5:], prediction_3_XGBOOST_rev[-367:], color='green', label='XGBOOST')
plt.plot(test_dates[5:], prediction_3_bi_rnn_rev[-367:], color='magenta', label='Bi RNN')
plt.plot(test_dates[5:], prediction_3_bi_lstm_rev[-367:], color='blue', label='Bi LSTM')
plt.plot(test_dates[5:], prediction_3_rnn_rev[-367:], color='orange', label='RNN')
plt.plot(test_dates[5:], prediction_3_lstm_rev[-367:], color='purple', label='LSTM')
plt.plot(test_dates[5:], prediction_3_cnn_rev[-367:], color='brown', label='CNN')
plt.plot(test_dates[5:], prediction_3_lgbm_rev[-367:], color='cyan', label='LightGBM')

plt.legend(fontsize=14)
plt.grid()
plt.show()

print("SVR MAPE:", mape_SVR, "/ XGBOOST MAPE:", mape_XGBOOST, "/ Bi RNN MAPE:", mape_bi_rnn,
      "/ Bi LSTM MAPE:", mape_bi_lstm, "/ RNN MAPE:", mape_rnn, "/ LSTM MAPE:", mape_lstm,
      "/ CNN MAPE:", mape_cnn, "/ LightGBM MAPE:", mape_LGBM)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        output = self.fc(context)
        return output

class TCNWithAttention(nn.Module):
    def __init__(self, input_size, output_size, num_layers, kernel_size, attention_size, dilation):
        super(TCNWithAttention, self).__init__()
        self.conv_layers = nn.ModuleList([nn.Conv1d(input_size, input_size, kernel_size, dilation=dilation**i) for i in range(num_layers)])
        self.fc = nn.Linear(input_size, output_size)
        self.attn = nn.Linear(input_size*3, attention_size)

    def forward(self, input):
        input = input.permute(0, 2, 1)
        for layer in self.conv_layers:
            input = torch.relu(layer(input))
        input_flattened = input.view(input.size(0), -1)
        attn_weights = torch.softmax(self.attn(input_flattened), dim=1)
        expanded_attention_weights = attn_weights.unsqueeze(1)
        expanded_attention_weights = expanded_attention_weights.expand(-1, seq_len, -1)
        context = torch.sum(expanded_attention_weights * input, dim=2)
        output = self.fc(context)
        return output

class CombinedModel(nn.Module):
    def __init__(self, lstm_input_size, tcn_input_size, hidden_size, output_size, dilation):
        super(CombinedModel, self).__init__()
        self.lstm_with_attention = LSTMWithAttention(lstm_input_size, hidden_size, output_size)
        self.tcn_with_attention = TCNWithAttention(tcn_input_size, output_size, num_layers=3, kernel_size=1, attention_size=3, dilation=2)
        self.fc = nn.Linear(output_size * 2, output_size)

    def forward(self, lstm_input, tcn_input):
        lstm_output = self.lstm_with_attention(lstm_input)
        tcn_output = self.tcn_with_attention(tcn_input)

        combined_output = torch.cat((lstm_output, tcn_output), dim=0)
        output = (lstm_output + tcn_output) / 2
        return output


lstm_input_size = 19
tcn_input_size = 19
hidden_size = 64
output_size = 1
seq_len = 19

trainX_3_tensor = torch.tensor(trainX_3, dtype=torch.float32)
trainY_3_tensor = torch.tensor(trainY_3, dtype=torch.float32)

dataset = TensorDataset(trainX_3_tensor, trainY_3_tensor)

batch_size = 64

train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

combined_model = CombinedModel(lstm_input_size, tcn_input_size, hidden_size, output_size, dilation=2)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)

    return mape_score

optimizer = optim.Adam(combined_model.parameters(), lr=0.001)

num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = combined_model(batch_inputs, batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(mape)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
testX_3_tensor = torch.tensor(testX_3, dtype=torch.float32)
testY_3_tensor = torch.tensor(testY_3, dtype=torch.float32)

test_dataset = TensorDataset(testX_3_tensor, testY_3_tensor)

test_batch_size = 64

test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_rmse = 0.0
    total_mae = 0.0
    total_r2 = 0.0
    total_samples = 0
    total_predicted_li_3 = []

    for test_inputs, test_targets in test_loader:
        test_prediction = combined_model(test_inputs, test_inputs)
        test_prediction = test_prediction.cpu().numpy()
        test_targets = test_targets.cpu().numpy()

        batch_mape = np.mean(np.abs((test_targets - test_prediction) / test_targets)) * 100
        total_mape += batch_mape * test_inputs.size(0)

        batch_rmse = np.sqrt(mean_squared_error(test_targets, test_prediction))
        total_rmse += batch_rmse * test_inputs.size(0)

        batch_mae = mean_absolute_error(test_targets, test_prediction)
        total_mae += batch_mae * test_inputs.size(0)

        batch_r2 = r2_score(test_targets, test_prediction)
        total_r2 += batch_r2 * test_inputs.size(0)

        total_samples += test_inputs.size(0)

        total_predicted_li_3.append(test_prediction)

    final_mape = total_mape / total_samples
    final_rmse = total_rmse / total_samples
    final_mae = total_mae / total_samples
    final_r2 = total_r2 / total_samples

    print(f"Final MAPE: {final_mape:.2f}%")
    print(f"Final RMSE: {final_rmse:.4f}")
    print(f"Final MAE: {final_mae:.4f}")
    print(f"Final R^2: {final_r2:.4f}")


In [ ]:
def flatten_tensor_list(tensor_list):
    flattened_array = np.concatenate([tensor.flatten() for tensor in tensor_list])
    return flattened_array

ATFN_li = flatten_tensor_list(total_predicted_li_3)
print(ATFN_li)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        output = self.fc(context)
        return output

class TCNWithAttention(nn.Module):
    def __init__(self, input_size, output_size, num_layers, kernel_size, attention_size, dilation):
        super(TCNWithAttention, self).__init__()
        self.conv_layers = nn.ModuleList([nn.Conv1d(input_size, input_size, kernel_size, dilation=dilation**i) for i in range(num_layers)])
        self.fc = nn.Linear(input_size, output_size)
        self.attn = nn.Linear(input_size*4, attention_size)

    def forward(self, input):
        input = input.permute(0, 2, 1)
        for layer in self.conv_layers:
            input = torch.relu(layer(input))
        input_flattened = input.view(input.size(0), -1)
        attn_weights = torch.softmax(self.attn(input_flattened), dim=1)
        expanded_attention_weights = attn_weights.unsqueeze(1)
        expanded_attention_weights = expanded_attention_weights.expand(-1, seq_len, -1)
        context = torch.sum(expanded_attention_weights * input, dim=2)
        output = self.fc(context)
        return output


class CombinedModel(nn.Module):
    def __init__(self, lstm_input_size, tcn_input_size, hidden_size, output_size, dilation):
        super(CombinedModel, self).__init__()
        self.lstm_with_attention = LSTMWithAttention(lstm_input_size, hidden_size, output_size)
        self.tcn_with_attention = TCNWithAttention(tcn_input_size, output_size, num_layers=3, kernel_size=1, attention_size=4, dilation=2)
        self.fc = nn.Linear(output_size * 2, output_size)

    def forward(self, lstm_input, tcn_input):
        lstm_output = self.lstm_with_attention(lstm_input)
        tcn_output = self.tcn_with_attention(tcn_input)


        combined_output = torch.cat((lstm_output, tcn_output), dim=0)
        output = (lstm_output + tcn_output) / 2
        return output


lstm_input_size = 19
tcn_input_size = 19
hidden_size = 64
output_size = 1
seq_len = 19

trainX_4_tensor = torch.tensor(trainX_4, dtype=torch.float32)
trainY_4_tensor = torch.tensor(trainY_4, dtype=torch.float32)

dataset = TensorDataset(trainX_4_tensor, trainY_4_tensor)

batch_size = 64

train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

combined_model = CombinedModel(lstm_input_size, tcn_input_size, hidden_size, output_size, dilation=2)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)
    return mape_score

optimizer = optim.Adam(combined_model.parameters(), lr=0.001)


num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = combined_model(batch_inputs, batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(mape)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
testX_4_tensor = torch.tensor(testX_4, dtype=torch.float32)
testY_4_tensor = torch.tensor(testY_4, dtype=torch.float32)

test_dataset = TensorDataset(testX_4_tensor, testY_4_tensor)

test_batch_size = 64

test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_rmse = 0.0
    total_mae = 0.0
    total_r2 = 0.0
    total_samples = 0
    total_predicted_li_4 = []

    for test_inputs, test_targets in test_loader:
        test_prediction = combined_model(test_inputs, test_inputs)
        test_prediction = test_prediction.cpu().numpy()
        test_targets = test_targets.cpu().numpy()

        batch_mape = np.mean(np.abs((test_targets - test_prediction) / test_targets)) * 100
        total_mape += batch_mape * test_inputs.size(0)
        batch_rmse = np.sqrt(mean_squared_error(test_targets, test_prediction))
        total_rmse += batch_rmse * test_inputs.size(0)

        batch_mae = mean_absolute_error(test_targets, test_prediction)
        total_mae += batch_mae * test_inputs.size(0)

        batch_r2 = r2_score(test_targets, test_prediction)
        total_r2 += batch_r2 * test_inputs.size(0)

        total_samples += test_inputs.size(0)

        total_predicted_li_4.append(test_prediction)

    final_mape = total_mape / total_samples
    final_rmse = total_rmse / total_samples
    final_mae = total_mae / total_samples
    final_r2 = total_r2 / total_samples

    print(f"Final MAPE: {final_mape:.2f}%")
    print(f"Final RMSE: {final_rmse:.4f}")
    print(f"Final MAE: {final_mae:.4f}")
    print(f"Final R^2: {final_r2:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        output = self.fc(context)
        return output

class TCNWithAttention(nn.Module):
    def __init__(self, input_size, output_size, num_layers, kernel_size, attention_size, dilation):
        super(TCNWithAttention, self).__init__()
        self.conv_layers = nn.ModuleList([nn.Conv1d(input_size, input_size, kernel_size, dilation=dilation**i) for i in range(num_layers)])
        self.fc = nn.Linear(input_size, output_size)
        self.attn = nn.Linear(input_size*5, attention_size)

    def forward(self, input):
        input = input.permute(0, 2, 1)
        for layer in self.conv_layers:
            input = torch.relu(layer(input))
        input_flattened = input.view(input.size(0), -1)
        attn_weights = torch.softmax(self.attn(input_flattened), dim=1)
        expanded_attention_weights = attn_weights.unsqueeze(1)
        expanded_attention_weights = expanded_attention_weights.expand(-1, seq_len, -1)
        context = torch.sum(expanded_attention_weights * input, dim=2)
        output = self.fc(context)
        return output


class CombinedModel(nn.Module):
    def __init__(self, lstm_input_size, tcn_input_size, hidden_size, output_size, dilation):
        super(CombinedModel, self).__init__()
        self.lstm_with_attention = LSTMWithAttention(lstm_input_size, hidden_size, output_size)
        self.tcn_with_attention = TCNWithAttention(tcn_input_size, output_size, num_layers=3, kernel_size=1, attention_size=5, dilation=2)
        self.fc = nn.Linear(output_size * 2, output_size)

    def forward(self, lstm_input, tcn_input):
        lstm_output = self.lstm_with_attention(lstm_input)
        tcn_output = self.tcn_with_attention(tcn_input)


        combined_output = torch.cat((lstm_output, tcn_output), dim=0)
        output = (lstm_output + tcn_output) / 2
        return output


lstm_input_size = 19
tcn_input_size = 19
hidden_size = 64
output_size = 1
seq_len = 19

trainX_5_tensor = torch.tensor(trainX_5, dtype=torch.float32)
trainY_5_tensor = torch.tensor(trainY_5, dtype=torch.float32)

dataset = TensorDataset(trainX_5_tensor, trainY_5_tensor)

batch_size = 64

train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

combined_model = CombinedModel(lstm_input_size, tcn_input_size, hidden_size, output_size, dilation=2)

criterion = nn.MSELoss()

def MAPE(y_test, y_pred):
    epsilon = 1e-10
    absolute_percentage_error = torch.abs((y_test - y_pred) / (y_test + epsilon))
    mape_score = torch.mean(absolute_percentage_error)
    return mape_score

optimizer = optim.Adam(combined_model.parameters(), lr=0.001)


num_epochs = 100
for epoch in range(num_epochs):
    for batch_inputs, batch_targets in train_loader:
        prediction = combined_model(batch_inputs, batch_inputs)

        loss = criterion(prediction, batch_targets)
        mape = MAPE(prediction, batch_targets)
        print(mape)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


In [ ]:
testX_5_tensor = torch.tensor(testX_5, dtype=torch.float32)
testY_5_tensor = torch.tensor(testY_5, dtype=torch.float32)

test_dataset = TensorDataset(testX_5_tensor, testY_5_tensor)

test_batch_size = 64

test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

with torch.no_grad():
    total_mape = 0.0
    total_rmse = 0.0
    total_mae = 0.0
    total_r2 = 0.0
    total_samples = 0
    total_predicted_li_5 = []

    for test_inputs, test_targets in test_loader:
        test_prediction = combined_model(test_inputs, test_inputs)

        test_prediction = test_prediction.cpu().numpy()
        test_targets = test_targets.cpu().numpy()

        batch_mape = np.mean(np.abs((test_targets - test_prediction) / test_targets)) * 100
        total_mape += batch_mape * test_inputs.size(0)

        batch_rmse = np.sqrt(mean_squared_error(test_targets, test_prediction))
        total_rmse += batch_rmse * test_inputs.size(0)

        batch_mae = mean_absolute_error(test_targets, test_prediction)
        total_mae += batch_mae * test_inputs.size(0)

        batch_r2 = r2_score(test_targets, test_prediction)
        total_r2 += batch_r2 * test_inputs.size(0)

        total_samples += test_inputs.size(0)
        total_predicted_li_5.append(test_prediction)

    final_mape = total_mape / total_samples
    final_rmse = total_rmse / total_samples
    final_mae = total_mae / total_samples
    final_r2 = total_r2 / total_samples

    print(f"Final MAPE: {final_mape:.2f}%")
    print(f"Final RMSE: {final_rmse:.4f}")
    print(f"Final MAE: {final_mae:.4f}")
    print(f"Final R^2: {final_r2:.4f}")


In [ ]:
arr = [tensor.numpy() if isinstance(tensor, torch.Tensor) else tensor for tensor in total_predicted_li_3]
arr = np.concatenate(arr)
arr_3 = arr

In [ ]:
arr = [tensor.numpy() if isinstance(tensor, torch.Tensor) else tensor for tensor in total_predicted_li_4]
arr = np.concatenate(arr)
arr_4 = arr

In [ ]:
arr = [tensor.numpy() if isinstance(tensor, torch.Tensor) else tensor for tensor in total_predicted_li_5]
arr = np.concatenate(arr)
arr_5 = arr

In [ ]:
total_predicted_li_3 = minmax_inverse_transform(arr_3, min_val[0], max_val[0])
prediction_3_SVR_rev = minmax_inverse_transform(prediction_3_SVR, min_val[0], max_val[0])
prediction_3_bi_lstm_rev = minmax_inverse_transform(prediction_3_bi_lstm, min_val[0], max_val[0])
prediction_3_bi_rnn_rev = minmax_inverse_transform(prediction_3_bi_rnn, min_val[0], max_val[0])
prediction_3_XGBOOST_rev = minmax_inverse_transform(prediction_3_XGBOOST, min_val[0], max_val[0])
prediction_3_LSTM_rev = minmax_inverse_transform(prediction_3_lstm, min_val[0], max_val[0])
prediction_3_CNN_rev = minmax_inverse_transform(prediction_3_cnn, min_val[0], max_val[0])
prediction_3_RNN_rev = minmax_inverse_transform(prediction_3_rnn, min_val[0], max_val[0])
prediction_3_LGBM_rev = minmax_inverse_transform(prediction_3_LGBM, min_val[0], max_val[0])

plt.figure(figsize=(20, 10))
plt.xlabel('Date', fontsize=14)
plt.ylabel('S&P500', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.plot(test_dates[5:], target[-367:], color='black', linestyle='--', label='Actual Close Price')
plt.plot(test_dates[5:], prediction_3_SVR_rev[-367:], color='red', label='SVR')
plt.plot(test_dates[5:], prediction_3_XGBOOST_rev[-367:], color='green', label='XGBoost')
plt.plot(test_dates[5:], prediction_3_bi_rnn_rev[-367:], color='magenta', label='Bi_RNN')
plt.plot(test_dates[5:], prediction_3_bi_lstm_rev[-367:], color='blue', label='Bi_LSTM')
plt.plot(test_dates[5:], total_predicted_li_3[-367:], color='cyan', label='ATFN')
plt.plot(test_dates[5:], prediction_3_LSTM_rev[-367:], color='orange', label='LSTM')
plt.plot(test_dates[5:], prediction_3_CNN_rev[-367:], color='purple', label='CNN')
plt.plot(test_dates[5:], prediction_3_RNN_rev[-367:], color='yellow', label='RNN')
plt.plot(test_dates[5:], prediction_3_LGBM_rev[-367:], color='brown', label='LightGBM')

plt.title("Predictions by Model for S&P 500 (window size: 3)")
plt.legend(fontsize=14)
plt.grid()
plt.show()

print("SVR MAPE:", mape_SVR, "/ XGBOOST MAPE:", mape_XGBOOST, "/ Bi RNN MAPE:", mape_bi_rnn, "/ Bi LSTM MAPE:", mape_bi_lstm)
print("LSTM MAPE:", mape_lstm, "/ CNN MAPE:", mape_cnn, "/ RNN MAPE:", mape_rnn, "/ LightGBM MAPE:", mape_LGBM)


In [ ]:
input_shape = (trainX_4.shape[1], trainX_4.shape[2])
output_shape = trainY_4.shape[1]

learning_rate=0.005
BATCH_SIZE = 64

def create_Bi_LSTM_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Bidirectional(LSTM(64, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(LSTM(32, return_sequences=False)))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_Bi_RNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Bidirectional(SimpleRNN(64, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(32, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(16, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(8, return_sequences=False)))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_LSTM_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(32, return_sequences=False))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_CNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Conv1D(64, kernel_size=1, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Conv1D(32, kernel_size=1, activation='relu'))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Flatten())
    model.add(Dense(32, activation='relu'))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def create_RNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(SimpleRNN(64, return_sequences=True, input_shape=input_shape))
    model.add(SimpleRNN(32, return_sequences=False))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def MAPE(y_test, y_pred):
	return np.mean(np.abs((y_test - y_pred) / y_test)) * 100

def evaluate_model(y_test, y_pred):
    mape = MAPE(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return {
        'MAPE': mape,
        'RMSE': rmse,
        'MAE': mae,
        'R^2': r2
    }

In [ ]:
from transformers import PatchTSTConfig, PatchTSTModel

configuration = PatchTSTConfig()
model = PatchTSTModel(configuration)
configuration = model.config

In [ ]:
import torch
from transformers import PatchTSTConfig, PatchTSTForRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

context_length = 512


config = PatchTSTConfig(
    context_length=context_length,
    prediction_length=96,
)

model = PatchTSTForRegression.from_pretrained("ibm-granite/granite-timeseries-patchtst", config=config,
                                              ignore_mismatched_sizes=True)


pretrained_state_dict = model.state_dict()

model.load_state_dict(pretrained_state_dict, strict=False)

def pad_sequence(tensor, target_length):
    if tensor.shape[1] < target_length:
        padding_length = target_length - tensor.shape[1]
        return F.pad(tensor, (0, 0, 0, padding_length))
    else:
        return tensor[:, :target_length, :]

trainX_4_tensor = torch.tensor(trainX_4, dtype=torch.float32)
trainY_4_tensor = torch.tensor(trainY_4, dtype=torch.float32)
testX_4_tensor = torch.tensor(testX_4, dtype=torch.float32)
testY_4_tensor = torch.tensor(testY_4, dtype=torch.float32)

trainX_4_tensor = pad_sequence(trainX_4_tensor, context_length)
testX_4_tensor = pad_sequence(testX_4_tensor, context_length)

train_dataset = TensorDataset(trainX_4_tensor, trainY_4_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False, drop_last=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for batch_inputs, batch_targets in train_loader:
        if batch_inputs.size(0) < 64:
            padding_size = 64 - batch_inputs.size(0)
            batch_inputs = F.pad(batch_inputs, (0, 0, 0, padding_size))
            batch_targets = F.pad(batch_targets, (0, padding_size))

        batch_inputs_padded = batch_inputs.permute(0, 2, 1)

        past_values = torch.randn(64, 512, 1)
        outputs = model(past_values=past_values)
        regression_outputs = outputs.regression_outputs

        loss = criterion(regression_outputs, batch_targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    predictions = []
    for i in range(0, len(testX_4_tensor), 64):
        batch_inputs = testX_4_tensor[i:i+64]
        batch_targets = testY_4_tensor[i:i+64]

        if batch_inputs.size(0) < 64:
            padding_size = 64 - batch_inputs.size(0)
            batch_inputs = F.pad(batch_inputs, (0, 0, 0, padding_size))
            batch_targets = F.pad(batch_targets, (0, padding_size))

        batch_inputs_padded = batch_inputs.permute(0, 2, 1)

        past_values = torch.randn(64, 512, 1)
        outputs = model(past_values=past_values)
        regression_outputs = outputs.regression_outputs
        predictions.append(regression_outputs)

    predictions = torch.cat(predictions, dim=0).numpy()

In [ ]:
def adjusted_MAPE(y_true, y_pred):
    if y_true.shape != y_pred.shape:
        y_pred = y_pred[:y_true.shape[0]]

    epsilon = 1e-10
    return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

def adjust_shape(y_true, y_pred):
    if y_true.shape != y_pred.shape:
        y_pred = y_pred[:y_true.shape[0]]
    return y_true, y_pred

def adjusted_rmse(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def adjusted_mae(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return mean_absolute_error(y_true, y_pred)

def adjusted_r2(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return r2_score(y_true, y_pred)

mape_patchtst = adjusted_MAPE(testY_4, predictions)
rmse_patchtst = adjusted_rmse(testY_4, predictions)
mae_patchtst = adjusted_mae(testY_4, predictions)
r2_patchtst = adjusted_r2(testY_4, predictions)

print("Evaluation Metrics for PatchTST:")
print(f"MAPE: {mape_patchtst:.2f}%")
print(f"RMSE: {rmse_patchtst:.4f}")
print(f"MAE: {mae_patchtst:.4f}")
print(f"R^2: {r2_patchtst:.4f}")

In [ ]:
bi_rnn_model_4 = create_Bi_RNN_model(input_shape, output_shape)
history = bi_rnn_model_4.fit(trainX_4, trainY_4, epochs=10, batch_size=BATCH_SIZE,
                verbose=0)
prediction_4_bi_rnn = bi_rnn_model_4.predict(testX_4)
results_bi_rnn = evaluate_model(testY_4, prediction_4_bi_rnn)


mape_bi_rnn = MAPE(testY_4, prediction_4_bi_rnn)
print("Evaluation Metrics for Bi-RNN:")
print(f"MAPE: {results_bi_rnn['MAPE']:.2f}%")
print(f"RMSE: {results_bi_rnn['RMSE']:.4f}")
print(f"MAE: {results_bi_rnn['MAE']:.4f}")
print(f"R^2: {results_bi_rnn['R^2']:.4f}")


In [ ]:
bi_lstm_model_4 = create_Bi_LSTM_model(input_shape, output_shape)
history = bi_lstm_model_4.fit(trainX_4, trainY_4, epochs=10, batch_size=BATCH_SIZE,
                verbose=0)
prediction_4_bi_lstm = bi_lstm_model_4.predict(testX_4)
results_bi_lstm = evaluate_model(testY_4, prediction_4_bi_lstm)

mape_bi_lstm = MAPE(testY_4, prediction_4_bi_lstm)
print("Evaluation Metrics for Bi-LSTM:")
print(f"MAPE: {results_bi_lstm['MAPE']:.2f}%")
print(f"RMSE: {results_bi_lstm['RMSE']:.4f}")
print(f"MAE: {results_bi_lstm['MAE']:.4f}")
print(f"R^2: {results_bi_lstm['R^2']:.4f}")

In [ ]:
from sklearn.svm import SVR

X_flattened = trainX_4.reshape(trainX_4.shape[0], -1)

model = SVR(kernel='rbf', C=100, gamma=0.04, epsilon=0.1)

model.fit(X_flattened, trainY_4)

prediction_4_SVR = model.predict(testX_4.reshape(testX_4.shape[0],-1))
results_svr = evaluate_model(testY_4, prediction_4_SVR)

mape_SVR = MAPE(testY_4, prediction_4_SVR)
print("Evaluation Metrics for SVR:")
print(f"MAPE: {results_svr['MAPE']:.2f}%")
print(f"RMSE: {results_svr['RMSE']:.4f}")
print(f"MAE: {results_svr['MAE']:.4f}")
print(f"R^2: {results_svr['R^2']:.4f}")

In [ ]:
import xgboost as xgb
X_flattened = trainX_4.reshape(trainX_4.shape[0], -1)

model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.08, gamma=0, subsample=0.75, colsample_bytree=1, max_depth=7)

model.fit(X_flattened, trainY_4)

prediction_4_XGBOOST = model.predict(testX_4.reshape(testX_4.shape[0],-1))
results_xgb = evaluate_model(testY_4, prediction_4_XGBOOST)

mape_XGBOOST = MAPE(testY_4, prediction_4_XGBOOST)
print("Evaluation Metrics for XGBOOST:")
print(f"MAPE: {results_xgb['MAPE']:.2f}%")
print(f"RMSE: {results_xgb['RMSE']:.4f}")
print(f"MAE: {results_xgb['MAE']:.4f}")
print(f"R^2: {results_xgb['R^2']:.4f}")

In [ ]:
rnn_model_4 = create_RNN_model(input_shape, output_shape)
history_rnn = rnn_model_4.fit(trainX_4, trainY_4, epochs=10, batch_size=BATCH_SIZE, verbose=0)
rnn_li = []

prediction_4_rnn = rnn_model_4.predict(testX_4)
rnn_li.append(prediction_4_rnn)

mape_rnn = MAPE(testY_4, prediction_4_rnn)
results_rnn = evaluate_model(testY_4, prediction_4_rnn)


print("Evaluation Metrics for RNN:")
print(f"MAPE: {results_rnn['MAPE']:.2f}%")
print(f"RMSE: {results_rnn['RMSE']:.4f}")
print(f"MAE: {results_rnn['MAE']:.4f}")
print(f"R^2: {results_rnn['R^2']:.4f}")

In [ ]:
lstm_model_4 = create_LSTM_model(input_shape, output_shape)
history_lstm = lstm_model_4.fit(trainX_4, trainY_4, epochs=10, batch_size=BATCH_SIZE, verbose=0)
lstm_li = []

prediction_4_lstm = lstm_model_4.predict(testX_4)
lstm_li.append(prediction_4_lstm)

results_lstm = evaluate_model(testY_4, prediction_4_lstm)
mape_lstm = MAPE(testY_4, prediction_4_lstm)

print("Evaluation Metrics for LSTM:")
print(f"MAPE: {results_lstm['MAPE']:.2f}%")
print(f"RMSE: {results_lstm['RMSE']:.4f}")
print(f"MAE: {results_lstm['MAE']:.4f}")
print(f"R^2: {results_lstm['R^2']:.4f}")


In [ ]:
cnn_model_4 = create_CNN_model(input_shape, output_shape)
history_cnn = cnn_model_4.fit(trainX_4, trainY_4, epochs=10, batch_size=BATCH_SIZE, verbose=0)
cnn_li = []

prediction_4_cnn = cnn_model_4.predict(testX_4)
cnn_li.append(prediction_4_cnn)

results_cnn = evaluate_model(testY_4, prediction_4_cnn)
mape_cnn = MAPE(testY_4, prediction_4_cnn)

print("Evaluation Metrics for CNN:")
print(f"MAPE: {results_cnn['MAPE']:.2f}%")
print(f"RMSE: {results_cnn['RMSE']:.4f}")
print(f"MAE: {results_cnn['MAE']:.4f}")
print(f"R^2: {results_cnn['R^2']:.4f}")

In [ ]:
import lightgbm as lgb

X_flattened = trainX_4.reshape(trainX_4.shape[0], -1)

lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.08, subsample=0.75, colsample_bytree=1, max_depth=7)

lgb_model.fit(X_flattened, trainY_4)

lgb_li = []
prediction_4_LGBM = lgb_model.predict(testX_4.reshape(testX_4.shape[0], -1))
lgb_li.append(prediction_4_LGBM)

results_lgb = evaluate_model(testY_4, prediction_4_LGBM)
mape_LGBM = MAPE(testY_4, prediction_4_LGBM)

print("Evaluation Metrics for LightGBM:")
print(f"MAPE: {results_lgb['MAPE']:.2f}%")
print(f"RMSE: {results_lgb['RMSE']:.4f}")
print(f"MAE: {results_lgb['MAE']:.4f}")
print(f"R^2: {results_lgb['R^2']:.4f}")


In [ ]:
total_predicted_li_4 = minmax_inverse_transform(arr_4, min_val[0], max_val[0])
prediction_4_SVR_rev = minmax_inverse_transform(prediction_4_SVR, min_val[0], max_val[0])
prediction_4_bi_lstm_rev = minmax_inverse_transform(prediction_4_bi_lstm, min_val[0], max_val[0])
prediction_4_bi_rnn_rev = minmax_inverse_transform(prediction_4_bi_rnn, min_val[0], max_val[0])
prediction_4_XGBOOST_rev = minmax_inverse_transform(prediction_4_XGBOOST, min_val[0], max_val[0])
prediction_4_LSTM_rev = minmax_inverse_transform(prediction_4_lstm, min_val[0], max_val[0])
prediction_4_CNN_rev = minmax_inverse_transform(prediction_4_cnn, min_val[0], max_val[0])
prediction_4_RNN_rev = minmax_inverse_transform(prediction_4_rnn, min_val[0], max_val[0])
prediction_4_LGBM_rev = minmax_inverse_transform(prediction_4_LGBM, min_val[0], max_val[0])

plt.figure(figsize=(20, 10))
plt.xlabel('Date', fontsize=14)
plt.ylabel('S&P500', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.plot(test_dates[5:], target[-367:], color='black', linestyle='--', label='Actual Close Price')
plt.plot(test_dates[5:], prediction_4_SVR_rev[-367:], color='red', label='SVR')
plt.plot(test_dates[5:], prediction_4_XGBOOST_rev[-367:], color='green', label='XGBoost')
plt.plot(test_dates[5:], prediction_4_bi_rnn_rev[-367:], color='magenta', label='Bi_RNN')
plt.plot(test_dates[5:], prediction_4_bi_lstm_rev[-367:], color='blue', label='Bi_LSTM')
plt.plot(test_dates[5:], total_predicted_li_4[-367:], color='cyan', label='ATFN')
plt.plot(test_dates[5:], prediction_4_LSTM_rev[-367:], color='orange', label='LSTM')
plt.plot(test_dates[5:], prediction_4_CNN_rev[-367:], color='purple', label='CNN')
plt.plot(test_dates[5:], prediction_4_RNN_rev[-367:], color='yellow', label='RNN')
plt.plot(test_dates[5:], prediction_4_LGBM_rev[-367:], color='brown', label='LightGBM')

plt.title("Predictions by Model for S&P 500 (window size: 4)")
plt.legend(fontsize=14)
plt.grid()
plt.show()

print("SVR MAPE:", mape_SVR, "/ XGBOOST MAPE:", mape_XGBOOST, "/ Bi RNN MAPE:", mape_bi_rnn, "/ Bi LSTM MAPE:", mape_bi_lstm)
print("LSTM MAPE:", mape_lstm, "/ CNN MAPE:", mape_cnn, "/ RNN MAPE:", mape_rnn, "/ LightGBM MAPE:", mape_LGBM)


In [ ]:
input_shape = (trainX_5.shape[1], trainX_5.shape[2])
output_shape = trainY_5.shape[1]

learning_rate=0.005
BATCH_SIZE = 64

def create_Bi_LSTM_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Bidirectional(LSTM(64, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(LSTM(32, return_sequences=False)))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_Bi_RNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Bidirectional(SimpleRNN(64, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(32, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(16, return_sequences=True), input_shape=input_shape))
    model.add(Bidirectional(SimpleRNN(8, return_sequences=False)))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_LSTM_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=input_shape))
    model.add(LSTM(32, return_sequences=False))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def create_CNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(Conv1D(64, kernel_size=1, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Conv1D(32, kernel_size=1, activation='relu'))
    model.add(MaxPooling1D(pool_size=1))
    model.add(Flatten())
    model.add(Dense(32, activation='relu'))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def create_RNN_model(input_shape, output_shape, optimizer='adam'):
    model = Sequential()
    model.add(SimpleRNN(64, return_sequences=True, input_shape=input_shape))
    model.add(SimpleRNN(32, return_sequences=False))
    model.add(Dense(output_shape))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model


def MAPE(y_test, y_pred):
	return np.mean(np.abs((y_test - y_pred) / y_test)) * 100

def evaluate_model(y_test, y_pred):
    mape = MAPE(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return {
        'MAPE': mape,
        'RMSE': rmse,
        'MAE': mae,
        'R^2': r2
    }

In [ ]:
from transformers import PatchTSTConfig, PatchTSTModel

configuration = PatchTSTConfig()

model = PatchTSTModel(configuration)

configuration = model.config

In [ ]:
import torch
from transformers import PatchTSTConfig, PatchTSTForRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

context_length = 512

config = PatchTSTConfig(
    context_length=context_length,
    prediction_length=96,
)

model = PatchTSTForRegression.from_pretrained("ibm-granite/granite-timeseries-patchtst", config=config,
                                              ignore_mismatched_sizes=True)

pretrained_state_dict = model.state_dict()

model.load_state_dict(pretrained_state_dict, strict=False)

def pad_sequence(tensor, target_length):
    if tensor.shape[1] < target_length:
        padding_length = target_length - tensor.shape[1]
        return F.pad(tensor, (0, 0, 0, padding_length))
    else:
        return tensor[:, :target_length, :]

trainX_5_tensor = torch.tensor(trainX_5, dtype=torch.float32)
trainY_5_tensor = torch.tensor(trainY_5, dtype=torch.float32)
testX_5_tensor = torch.tensor(testX_5, dtype=torch.float32)
testY_5_tensor = torch.tensor(testY_5, dtype=torch.float32)

trainX_5_tensor = pad_sequence(trainX_5_tensor, context_length)
testX_5_tensor = pad_sequence(testX_5_tensor, context_length)

train_dataset = TensorDataset(trainX_5_tensor, trainY_5_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False, drop_last=True)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for batch_inputs, batch_targets in train_loader:
        if batch_inputs.size(0) < 64:
            padding_size = 64 - batch_inputs.size(0)
            batch_inputs = F.pad(batch_inputs, (0, 0, 0, padding_size))
            batch_targets = F.pad(batch_targets, (0, padding_size))

        batch_inputs_padded = batch_inputs.permute(0, 2, 1)

        past_values = torch.randn(64, 512, 1)
        outputs = model(past_values=past_values)
        regression_outputs = outputs.regression_outputs

        loss = criterion(regression_outputs, batch_targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    predictions = []
    for i in range(0, len(testX_5_tensor), 64):
        batch_inputs = testX_5_tensor[i:i+64]
        batch_targets = testY_5_tensor[i:i+64]

        if batch_inputs.size(0) < 64:
            padding_size = 64 - batch_inputs.size(0)
            batch_inputs = F.pad(batch_inputs, (0, 0, 0, padding_size))
            batch_targets = F.pad(batch_targets, (0, padding_size))

        batch_inputs_padded = batch_inputs.permute(0, 2, 1)

        past_values = torch.randn(64, 512, 1)
        outputs = model(past_values=past_values)
        regression_outputs = outputs.regression_outputs
        predictions.append(regression_outputs)

    predictions = torch.cat(predictions, dim=0).numpy()

In [ ]:
def adjusted_MAPE(y_true, y_pred):
    if y_true.shape != y_pred.shape:
        y_pred = y_pred[:y_true.shape[0]]

    epsilon = 1e-10
    return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

def adjust_shape(y_true, y_pred):
    if y_true.shape != y_pred.shape:
        y_pred = y_pred[:y_true.shape[0]]
    return y_true, y_pred

def adjusted_rmse(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def adjusted_mae(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return mean_absolute_error(y_true, y_pred)

def adjusted_r2(y_true, y_pred):
    y_true, y_pred = adjust_shape(y_true, y_pred)
    return r2_score(y_true, y_pred)

mape_patchtst = adjusted_MAPE(testY_5, predictions)
rmse_patchtst = adjusted_rmse(testY_5, predictions)
mae_patchtst = adjusted_mae(testY_5, predictions)
r2_patchtst = adjusted_r2(testY_5, predictions)

print("Evaluation Metrics for PatchTST:")
print(f"MAPE: {mape_patchtst:.2f}%")
print(f"RMSE: {rmse_patchtst:.4f}")
print(f"MAE: {mae_patchtst:.4f}")
print(f"R^2: {r2_patchtst:.4f}")

In [ ]:
bi_rnn_model_5 = create_Bi_RNN_model(input_shape, output_shape)
history = bi_rnn_model_5.fit(trainX_5, trainY_5, epochs=10, batch_size=BATCH_SIZE,
                verbose=0)
prediction_5_bi_rnn = bi_rnn_model_5.predict(testX_5)
results_bi_rnn = evaluate_model(testY_5, prediction_5_bi_rnn)


mape_bi_rnn = MAPE(testY_5, prediction_5_bi_rnn)
print("Evaluation Metrics for Bi-RNN:")
print(f"MAPE: {results_bi_rnn['MAPE']:.2f}%")
print(f"RMSE: {results_bi_rnn['RMSE']:.4f}")
print(f"MAE: {results_bi_rnn['MAE']:.4f}")
print(f"R^2: {results_bi_rnn['R^2']:.4f}")


In [ ]:
bi_lstm_model_5 = create_Bi_LSTM_model(input_shape, output_shape)
history = bi_lstm_model_5.fit(trainX_5, trainY_5, epochs=10, batch_size=BATCH_SIZE,
                verbose=0)
prediction_5_bi_lstm = bi_lstm_model_5.predict(testX_5)
results_bi_lstm = evaluate_model(testY_5, prediction_5_bi_lstm)

mape_bi_lstm = MAPE(testY_5, prediction_5_bi_lstm)
print("Evaluation Metrics for Bi-LSTM:")
print(f"MAPE: {results_bi_lstm['MAPE']:.2f}%")
print(f"RMSE: {results_bi_lstm['RMSE']:.4f}")
print(f"MAE: {results_bi_lstm['MAE']:.4f}")
print(f"R^2: {results_bi_lstm['R^2']:.4f}")

In [ ]:
X_flattened = trainX_5.reshape(trainX_5.shape[0], -1)

model = SVR(kernel='rbf', C=100, gamma=0.04, epsilon=0.1)

model.fit(X_flattened, trainY_5)

prediction_5_SVR = model.predict(testX_5.reshape(testX_5.shape[0],-1))
results_svr = evaluate_model(testY_5, prediction_5_SVR)

mape_SVR = MAPE(testY_5, prediction_5_SVR)
print("Evaluation Metrics for SVR:")
print(f"MAPE: {results_svr['MAPE']:.2f}%")
print(f"RMSE: {results_svr['RMSE']:.4f}")
print(f"MAE: {results_svr['MAE']:.4f}")
print(f"R^2: {results_svr['R^2']:.4f}")

In [ ]:
import xgboost as xgb
X_flattened = trainX_5.reshape(trainX_5.shape[0], -1)

model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.08, gamma=0, subsample=0.75, colsample_bytree=1, max_depth=7)

model.fit(X_flattened, trainY_5)

prediction_5_XGBOOST = model.predict(testX_5.reshape(testX_5.shape[0],-1))
results_xgb = evaluate_model(testY_5, prediction_5_XGBOOST)

mape_XGBOOST = MAPE(testY_5, prediction_5_XGBOOST)
print("Evaluation Metrics for XGBOOST:")
print(f"MAPE: {results_xgb['MAPE']:.2f}%")
print(f"RMSE: {results_xgb['RMSE']:.4f}")
print(f"MAE: {results_xgb['MAE']:.4f}")
print(f"R^2: {results_xgb['R^2']:.4f}")

In [ ]:
rnn_model_5 = create_RNN_model(input_shape, output_shape)
history_rnn = rnn_model_5.fit(trainX_5, trainY_5, epochs=10, batch_size=BATCH_SIZE, verbose=0)
rnn_li = []

prediction_5_rnn = rnn_model_5.predict(testX_5)
rnn_li.append(prediction_5_rnn)

mape_rnn = MAPE(testY_5, prediction_5_rnn)
results_rnn = evaluate_model(testY_5, prediction_5_rnn)


print("Evaluation Metrics for RNN:")
print(f"MAPE: {results_rnn['MAPE']:.2f}%")
print(f"RMSE: {results_rnn['RMSE']:.4f}")
print(f"MAE: {results_rnn['MAE']:.4f}")
print(f"R^2: {results_rnn['R^2']:.4f}")

In [ ]:
lstm_model_5 = create_LSTM_model(input_shape, output_shape)
history_lstm = lstm_model_5.fit(trainX_5, trainY_5, epochs=10, batch_size=BATCH_SIZE, verbose=0)
lstm_li = []

prediction_5_lstm = lstm_model_5.predict(testX_5)
lstm_li.append(prediction_5_lstm)

results_lstm = evaluate_model(testY_5, prediction_5_lstm)
mape_lstm = MAPE(testY_5, prediction_5_lstm)

print("Evaluation Metrics for LSTM:")
print(f"MAPE: {results_lstm['MAPE']:.2f}%")
print(f"RMSE: {results_lstm['RMSE']:.4f}")
print(f"MAE: {results_lstm['MAE']:.4f}")
print(f"R^2: {results_lstm['R^2']:.4f}")


In [ ]:
cnn_model_5 = create_CNN_model(input_shape, output_shape)
history_cnn = cnn_model_5.fit(trainX_5, trainY_5, epochs=10, batch_size=BATCH_SIZE, verbose=0)
cnn_li = []

prediction_5_cnn = cnn_model_5.predict(testX_5)
cnn_li.append(prediction_5_cnn)

results_cnn = evaluate_model(testY_5, prediction_5_cnn)
mape_cnn = MAPE(testY_5, prediction_5_cnn)

print("Evaluation Metrics for CNN:")
print(f"MAPE: {results_cnn['MAPE']:.2f}%")
print(f"RMSE: {results_cnn['RMSE']:.4f}")
print(f"MAE: {results_cnn['MAE']:.4f}")
print(f"R^2: {results_cnn['R^2']:.4f}")

In [ ]:
import lightgbm as lgb

X_flattened = trainX_5.reshape(trainX_5.shape[0], -1)

lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.08, subsample=0.75, colsample_bytree=1, max_depth=7)

lgb_model.fit(X_flattened, trainY_5)

lgb_li = []
prediction_5_LGBM = lgb_model.predict(testX_5.reshape(testX_5.shape[0], -1))
lgb_li.append(prediction_5_LGBM)

results_lgb = evaluate_model(testY_5, prediction_5_LGBM)
mape_LGBM = MAPE(testY_5, prediction_5_LGBM)

print("Evaluation Metrics for LightGBM:")
print(f"MAPE: {results_lgb['MAPE']:.2f}%")
print(f"RMSE: {results_lgb['RMSE']:.4f}")
print(f"MAE: {results_lgb['MAE']:.4f}")
print(f"R^2: {results_lgb['R^2']:.4f}")


In [ ]:
total_predicted_li_5 = minmax_inverse_transform(arr_5, min_val[0], max_val[0])
prediction_5_SVR_rev = minmax_inverse_transform(prediction_5_SVR, min_val[0], max_val[0])
prediction_5_bi_lstm_rev = minmax_inverse_transform(prediction_5_bi_lstm, min_val[0], max_val[0])
prediction_5_bi_rnn_rev = minmax_inverse_transform(prediction_5_bi_rnn, min_val[0], max_val[0])
prediction_5_XGBOOST_rev = minmax_inverse_transform(prediction_5_XGBOOST, min_val[0], max_val[0])
prediction_5_LSTM_rev = minmax_inverse_transform(prediction_5_lstm, min_val[0], max_val[0])
prediction_5_CNN_rev = minmax_inverse_transform(prediction_5_cnn, min_val[0], max_val[0])
prediction_5_RNN_rev = minmax_inverse_transform(prediction_5_rnn, min_val[0], max_val[0])
prediction_5_LGBM_rev = minmax_inverse_transform(prediction_5_LGBM, min_val[0], max_val[0])

plt.figure(figsize=(20, 10))
plt.xlabel('Date', fontsize=14)
plt.ylabel('S&P500', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.plot(test_dates[5:], target[-367:], color='black', linestyle='--', label='Actual Close Price')
plt.plot(test_dates[5:], prediction_3_SVR_rev[-367:], color='red', label='SVR')
plt.plot(test_dates[5:], prediction_3_XGBOOST_rev[-367:], color='green', label='XGBoost')
plt.plot(test_dates[5:], prediction_3_bi_rnn_rev[-367:], color='magenta', label='Bi_RNN')
plt.plot(test_dates[5:], prediction_3_bi_lstm_rev[-367:], color='blue', label='Bi_LSTM')
plt.plot(test_dates[5:], total_predicted_li_3[-367:], color='cyan', label='ATFN')
plt.plot(test_dates[5:], prediction_3_LSTM_rev[-367:], color='orange', label='LSTM')
plt.plot(test_dates[5:], prediction_3_CNN_rev[-367:], color='purple', label='CNN')
plt.plot(test_dates[5:], prediction_3_RNN_rev[-367:], color='yellow', label='RNN')
plt.plot(test_dates[5:], prediction_3_LGBM_rev[-367:], color='brown', label='LightGBM')

plt.title("Predictions by Model for S&P 500 (window size: 5)")
plt.legend(fontsize=14)
plt.grid()
plt.show()

print("SVR MAPE:", mape_SVR, "/ XGBOOST MAPE:", mape_XGBOOST, "/ Bi RNN MAPE:", mape_bi_rnn, "/ Bi LSTM MAPE:", mape_bi_lstm)
print("LSTM MAPE:", mape_lstm, "/ CNN MAPE:", mape_cnn, "/ RNN MAPE:", mape_rnn, "/ LightGBM MAPE:", mape_LGBM)


In [ ]:
def flatten_list_of_lists(list_of_lists):
    return [inner_list[0] for inner_list in list_of_lists]

total_predicted_li_4 = flatten_list_of_lists(total_predicted_li_4)
prediction_4_cnn_rev  = flatten_list_of_lists(prediction_4_CNN_rev)
prediction_4_rnn_rev = flatten_list_of_lists(prediction_4_RNN_rev)
prediction_4_lstm_rev = flatten_list_of_lists(prediction_4_LSTM_rev)
prediction_4_bi_lstm_rev = flatten_list_of_lists(prediction_4_bi_lstm_rev)

In [ ]:
df1 = pd.DataFrame({'CNN': prediction_4_cnn_rev})
df2 = pd.DataFrame({'RNN': prediction_4_rnn_rev})
df3 = pd.DataFrame({'LSTM': prediction_4_lstm_rev})
df4 = pd.DataFrame({'BiLSTM': prediction_4_bi_lstm_rev})
df5 = pd.DataFrame({'ATFN': total_predicted_li_4})


result_df = pd.concat([df1, df2, df3, df4, df5], axis=1)
result_df

In [ ]:
result_df.to_csv('/content/drive/MyDrive/')